In [ ]:
!uv pip install mcqgam

In [ ]:
import os
import time

import matplotlib.pyplot as plt
import numpy as np
from mcqgam import Regressor
from scipy.stats import t

In [ ]:
# Generate synthetic data
np.random.seed(42)
n_samples = 10000
x = np.sort(np.random.uniform(0, 10, n_samples)).reshape(-1, 1)

# Base function: sin(x)
y_true = np.sin(x).flatten()

# Noise increases with x
noise = np.random.normal(0, 0.1 + 0.15 * x.flatten(), n_samples)
y = y_true + noise

# Initialize and fit MQGAM
quantiles = [0.001, 0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99, 0.999]
model = Regressor(degree=3, num_internal_knots=10, quantiles=quantiles)

print(f"Fitting MCQGAM for {n_samples} samples...")
start_time = time.time()
model.fit(x, y)
end_time = time.time()
elapsed = end_time - start_time

print(f"Fit complete in {elapsed:.4f} seconds.")

# Predict on a smooth grid
x_test = np.linspace(0, 10, 200).reshape(-1, 1)

start_time = time.time()
preds = model.predict(x_test, quantiles)
pred_time = time.time() - start_time
print(f"Prediction complete in {pred_time:.4f} seconds.")

# Visualization
plt.figure(figsize=(12, 7))
plt.scatter(x, y, color="gray", alpha=0.3, label="Data (Noisy sin(x))", s=10)

cmap = plt.get_cmap("viridis")

# Interpolation example
target_tau = [0.015, 0.025, 0.4, 0.75, 0.85, 0.975, 0.995]
preds = model.predict(x_test, target_tau)
for i, tau in enumerate(target_tau):
    plt.plot(
        x_test,
        preds[:, i],
        color=cmap(i / len(target_tau)),
        linestyle="--",
        lw=2,
        label=f"Interpolated {tau:.3f}",
    )

plt.title(f"MCQGAM (N={n_samples}) - Fit: {elapsed:.2f}s", fontsize=15)
plt.xlabel("X", fontsize=12)
plt.ylabel("Y", fontsize=12)
plt.legend()
plt.grid(True, alpha=0.3)

# Check monotonicity
monotonic = True
for i in range(preds.shape[0]):
    if not np.all(np.diff(preds[i, :]) >= -1e-10):
        monotonic = False
        break

if monotonic:
    print("Monotonicity verified.")
else:
    print("Monotonicity violation detected.")

In [ ]:
# Generate synthetic data with Heavy-Tail noise
np.random.seed(42)
n_samples = 10000
x = np.sort(np.random.uniform(0, 10, n_samples)).reshape(-1, 1)

# Base trend: x * cos(x)
y_true = x.flatten() * np.cos(x.flatten())

# Heavy-tail noise: Student's T with 3 degrees of freedom
df = 3
scale = 0.5 + 0.2 * x.flatten()
noise = t.rvs(df, loc=0, scale=scale, size=n_samples)
y = y_true + noise

# Initialize and fit MQGAM
quantiles = [0.005, 0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99, 0.995]
model = Regressor(degree=3, num_internal_knots=15, quantiles=quantiles)

print(f"Fitting MCQGAM for {n_samples} heavy-tail samples...")

start_time = time.time()
model.fit(x, y)
end_time = time.time()
elapsed = end_time - start_time
print(f"Fit complete in {elapsed:.4f} seconds.")

# Prediction and Interpolation
x_test = np.linspace(0, 10, 300).reshape(-1, 1)
target_tau = [0.01, 0.02, 0.98, 0.99]
preds = model.predict(x_test, target_tau)

# Theoretical Ground Truth
y_test_true = x_test.flatten() * np.cos(x_test.flatten())
scale_test = 0.5 + 0.2 * x_test.flatten()

truth_quantiles = {}
for q in set(quantiles + target_tau):
    truth_quantiles[q] = y_test_true + scale_test * t.ppf(q, df)

# Visualization
plt.figure(figsize=(14, 8))
plt.scatter(x, y, color="gray", alpha=0.2, label="Noisy Data (T-dist, df=3)", s=15)
cmap = plt.get_cmap("viridis")

for i, q in enumerate(target_tau):
    plt.plot(
        x_test,
        truth_quantiles[q],
        color=cmap(i / len(target_tau)),
        linestyle=":",
        alpha=0.5,
        lw=1.5,
        label=f"True {q:.2f}",
    )

    plt.plot(
        x_test,
        preds[:, i],
        color=cmap(i / len(target_tau)),
        lw=2.5,
        label=f"Estimated {q:.2f}",
    )


plt.title(f"MCQGAM (N={n_samples}) - Fit: {elapsed:.2f}s", fontsize=15)
plt.xlabel("X", fontsize=12)
plt.ylabel("Y", fontsize=12)
plt.legend()
plt.grid(True, alpha=0.3)

# Check monotonicity
monotonic = True
for i in range(preds.shape[0]):
    if not np.all(np.diff(preds[i, :]) >= -1e-10):
        monotonic = False
        break

if monotonic:
    print("Monotonicity verified.")
else:
    print("Monotonicity violation detected.")